In [1]:
# ============================================================
# NOTEBOOK 03: MODELADO
# Proyecto: Credit Risk Scoring ML
# Autor: Marín Serrato Barrios
# Descripción: Entrenamiento y evaluación del modelo
#              de scoring crediticio con LightGBM.
#              Basado en features del notebook 02.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Ruta base del proyecto
RUTA_PROYECTO = "c:/Users/Marin/Documents/PROYECTO ML_OPS/credit-risk-scoring-ml"
os.chdir(RUTA_PROYECTO)

print(f"Directorio: {os.getcwd()}")
print("\n✅ Configuración correcta")

Directorio: c:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml

✅ Configuración correcta


In [2]:
# --- CARGA DE DATASETS PROCESADOS ---

# Cargamos los datasets que preparamos en el Feature Engineering
# Ya están limpios, con features derivados y encoding aplicado

print("CARGANDO DATASETS PROCESADOS...")
print("="*60)

# Dataset para LightGBM
df_lgbm = pd.read_csv("data/processed/df_lgbm.csv")

# Dataset para Scorecard
df_scorecard = pd.read_csv("data/processed/df_scorecard.csv")

print(f"df_lgbm:")
print(f"  Shape:  {df_lgbm.shape}")
print(f"  Mora:   {df_lgbm['TARGET'].mean()*100:.2f}%")

print(f"\ndf_scorecard:")
print(f"  Shape:  {df_scorecard.shape}")
print(f"  Mora:   {df_scorecard['TARGET'].mean()*100:.2f}%")

# Separar features y target
X_lgbm     = df_lgbm.drop(columns=['TARGET', 'SK_ID_CURR'])
y          = df_lgbm['TARGET']

X_scorecard = df_scorecard.drop(columns=['TARGET', 'SK_ID_CURR'])
y_score     = df_scorecard['TARGET']

print(f"\nFeatures LightGBM:  {X_lgbm.shape[1]}")
print(f"Features Scorecard: {X_scorecard.shape[1]}")
print(f"Observaciones:      {len(y):,}")
print(f"\n✅ Datos cargados correctamente")

CARGANDO DATASETS PROCESADOS...
df_lgbm:
  Shape:  (307511, 67)
  Mora:   8.07%

df_scorecard:
  Shape:  (307511, 124)
  Mora:   8.07%

Features LightGBM:  65
Features Scorecard: 122
Observaciones:      307,511

✅ Datos cargados correctamente


In [3]:
# --- PASO 2: SPLIT TEMPORAL TRAIN / VALIDATION / TEST ---

# REGLA FUNDAMENTAL DEL FRAMEWORK:
# Nunca validación cruzada aleatoria en datos de crédito.
# Siempre validación temporal que respeta el orden cronológico.
#
# El dataset de Home Credit tiene una variable implícita de tiempo:
# SK_ID_CURR es un identificador secuencial.
# IDs más altos = solicitudes más recientes.
# Usamos esto para simular un split temporal real.

print("PASO 2: SPLIT TEMPORAL TRAIN / VALIDATION / TEST")
print("="*60)

# Ordenar por SK_ID_CURR (proxy de tiempo)
idx_ordenado = df_lgbm['SK_ID_CURR'].argsort().values

# Proporciones del split
# 70% entrenamiento, 15% validación, 15% test
n_total = len(y)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)

# Índices temporales
idx_train = idx_ordenado[:n_train]
idx_val   = idx_ordenado[n_train:n_train + n_val]
idx_test  = idx_ordenado[n_train + n_val:]

# Split para LightGBM
X_train = X_lgbm.iloc[idx_train]
X_val   = X_lgbm.iloc[idx_val]
X_test  = X_lgbm.iloc[idx_test]

y_train = y.iloc[idx_train]
y_val   = y.iloc[idx_val]
y_test  = y.iloc[idx_test]

# Split para Scorecard
X_train_sc = X_scorecard.iloc[idx_train]
X_val_sc   = X_scorecard.iloc[idx_val]
X_test_sc  = X_scorecard.iloc[idx_test]

y_train_sc = y_score.iloc[idx_train]
y_val_sc   = y_score.iloc[idx_val]
y_test_sc  = y_score.iloc[idx_test]

print(f"ENTRENAMIENTO: {len(y_train):>8,} registros "
      f"({len(y_train)/n_total*100:.1f}%) "
      f"mora: {y_train.mean()*100:.2f}%")
print(f"VALIDACIÓN:    {len(y_val):>8,} registros "
      f"({len(y_val)/n_total*100:.1f}%) "
      f"mora: {y_val.mean()*100:.2f}%")
print(f"TEST:          {len(y_test):>8,} registros "
      f"({len(y_test)/n_total*100:.1f}%) "
      f"mora: {y_test.mean()*100:.2f}%")

print()
print("VERIFICACIÓN DE CONSISTENCIA:")
print(f"  Tasa mora global:       {y.mean()*100:.2f}%")
print(f"  Tasa mora entrenamiento:{y_train.mean()*100:.2f}%")
print(f"  Tasa mora validación:   {y_val.mean()*100:.2f}%")
print(f"  Tasa mora test:         {y_test.mean()*100:.2f}%")
print()
print("  Si las tasas son similares entre particiones,")
print("  el split es representativo ✅")

PASO 2: SPLIT TEMPORAL TRAIN / VALIDATION / TEST
ENTRENAMIENTO:  215,257 registros (70.0%) mora: 8.13%
VALIDACIÓN:      46,126 registros (15.0%) mora: 7.98%
TEST:            46,128 registros (15.0%) mora: 7.91%

VERIFICACIÓN DE CONSISTENCIA:
  Tasa mora global:       8.07%
  Tasa mora entrenamiento:8.13%
  Tasa mora validación:   7.98%
  Tasa mora test:         7.91%

  Si las tasas son similares entre particiones,
  el split es representativo ✅


In [6]:
# --- IMPUTACIÓN DE NaN EN df_scorecard ---

# El CSV se guardó con NaN residuales.
# Los imputamos aquí antes de entrenar.

print("IMPUTANDO NaN EN df_scorecard...")
print("="*60)

# Variables numéricas: imputar con mediana
cols_mediana = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
    'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
    'DAYS_LAST_PHONE_CHANGE',
    'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR',
    'ANTIGUEDAD_LABORAL_AÑOS',
    'EXT_SOURCE_PROMEDIO', 'EXT_SOURCE_MIN',
    'DTI_X_EXT2', 'RIESGO_EDAD_SCORE'
]

# Variables con imputación especial
# OWN_CAR_AGE: 0 si no tiene auto
cols_cero = ['OWN_CAR_AGE']

# Imputar en el dataset completo ANTES del split
# para que la mediana sea consistente
for col in cols_mediana:
    if col in df_scorecard.columns:
        mediana = df_scorecard[col].median()
        df_scorecard[col] = df_scorecard[col].fillna(mediana)
        print(f"  {col:<35} → mediana: {mediana:.4f}")

for col in cols_cero:
    if col in df_scorecard.columns:
        df_scorecard[col] = df_scorecard[col].fillna(0)
        print(f"  {col:<35} → 0 (sin auto)")

# Verificar que no quedan NaN
X_scorecard = df_scorecard.drop(columns=['TARGET', 'SK_ID_CURR'])
print(f"\nNaN restantes: {X_scorecard.isnull().sum().sum()} ✅")

# Re-hacer el split con datos limpios
X_train_sc = X_scorecard.iloc[idx_train]
X_val_sc   = X_scorecard.iloc[idx_val]
X_test_sc  = X_scorecard.iloc[idx_test]

y_train_sc = y_score.iloc[idx_train]
y_val_sc   = y_score.iloc[idx_val]
y_test_sc  = y_score.iloc[idx_test]

print(f"Split rehecho correctamente ✅")
print(f"X_train_sc shape: {X_train_sc.shape}")

IMPUTANDO NaN EN df_scorecard...
  EXT_SOURCE_1                        → mediana: 0.5060
  EXT_SOURCE_2                        → mediana: 0.5660
  EXT_SOURCE_3                        → mediana: 0.5353
  OBS_30_CNT_SOCIAL_CIRCLE            → mediana: 0.0000
  DEF_30_CNT_SOCIAL_CIRCLE            → mediana: 0.0000
  OBS_60_CNT_SOCIAL_CIRCLE            → mediana: 0.0000
  DEF_60_CNT_SOCIAL_CIRCLE            → mediana: 0.0000
  DAYS_LAST_PHONE_CHANGE              → mediana: -757.0000
  AMT_REQ_CREDIT_BUREAU_DAY           → mediana: 0.0000
  AMT_REQ_CREDIT_BUREAU_MON           → mediana: 0.0000
  AMT_REQ_CREDIT_BUREAU_QRT           → mediana: 0.0000
  AMT_REQ_CREDIT_BUREAU_YEAR          → mediana: 1.0000
  ANTIGUEDAD_LABORAL_AÑOS             → mediana: 4.5151
  EXT_SOURCE_PROMEDIO                 → mediana: 0.5245
  EXT_SOURCE_MIN                      → mediana: 0.4032
  DTI_X_EXT2                          → mediana: 0.0724
  RIESGO_EDAD_SCORE                   → mediana: 0.2374
  OWN_CAR_AG

In [7]:
# --- PASO 3: SCORECARD LOGÍSTICO (BASELINE) ---

# La regresión logística es el modelo interpretable
# obligatorio en scoring crediticio.
# Lo entrenamos primero para establecer el benchmark
# que LightGBM debe superar.

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

print("PASO 3: SCORECARD LOGÍSTICO")
print("="*60)

# IMPORTANTE: La regresión logística requiere
# que las variables estén en la misma escala.
# StandardScaler normaliza a media=0 y desviación=1.
# LightGBM NO necesita esto (es invariante a escala).

# Pipeline: escalado + regresión logística
# El pipeline garantiza que el escalado se aprende
# SOLO en train y se aplica igual en val y test
# (evita data leakage en el preprocesamiento)

pipeline_scorecard = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(
        C=0.01,              # regularización fuerte
                             # evita overfitting con 122 features
        max_iter=1000,       # suficientes iteraciones para converger
        random_state=42,
        class_weight='balanced',  # maneja el desbalance 91/9
        solver='lbfgs',      # eficiente para datasets grandes
        n_jobs=-1            # usar todos los núcleos disponibles
    ))
])

print("Entrenando scorecard logístico...")
print("(puede tardar 1-2 minutos con 122 features)")
print()

pipeline_scorecard.fit(X_train_sc, y_train_sc)

# Probabilidades predichas
proba_train_sc = pipeline_scorecard.predict_proba(X_train_sc)[:,1]
proba_val_sc   = pipeline_scorecard.predict_proba(X_val_sc)[:,1]
proba_test_sc  = pipeline_scorecard.predict_proba(X_test_sc)[:,1]

# Métricas
auc_train_sc = roc_auc_score(y_train_sc, proba_train_sc)
auc_val_sc   = roc_auc_score(y_val_sc,   proba_val_sc)
auc_test_sc  = roc_auc_score(y_test_sc,  proba_test_sc)

# KS
def calcular_ks(y_true, y_proba):
    df_ks = pd.DataFrame({'real': y_true, 'proba': y_proba})
    df_ks = df_ks.sort_values('proba', ascending=False).reset_index(drop=True)
    df_ks['cum_malos']  = (df_ks['real']==1).cumsum() / df_ks['real'].sum()
    df_ks['cum_buenos'] = (df_ks['real']==0).cumsum() / (df_ks['real']==0).sum()
    return (df_ks['cum_malos'] - df_ks['cum_buenos']).max()

ks_train_sc = calcular_ks(y_train_sc, proba_train_sc)
ks_val_sc   = calcular_ks(y_val_sc,   proba_val_sc)
ks_test_sc  = calcular_ks(y_test_sc,  proba_test_sc)

print("RESULTADOS SCORECARD LOGÍSTICO:")
print(f"{'':20} {'TRAIN':>10} {'VAL':>10} {'TEST':>10}")
print("-"*52)
print(f"{'AUC':20} {auc_train_sc:>10.4f} {auc_val_sc:>10.4f} {auc_test_sc:>10.4f}")
print(f"{'KS':20} {ks_train_sc:>10.4f} {ks_val_sc:>10.4f} {ks_test_sc:>10.4f}")
print(f"{'Gini':20} {2*auc_train_sc-1:>10.4f} {2*auc_val_sc-1:>10.4f} {2*auc_test_sc-1:>10.4f}")
print()
print("DIAGNÓSTICO:")
brecha_auc = auc_train_sc - auc_val_sc
if brecha_auc < 0.02:
    print(f"  Brecha train-val AUC: {brecha_auc:.4f} ✅ Sin overfitting")
else:
    print(f"  Brecha train-val AUC: {brecha_auc:.4f} ⚠️  Posible overfitting")

PASO 3: SCORECARD LOGÍSTICO
Entrenando scorecard logístico...
(puede tardar 1-2 minutos con 122 features)

RESULTADOS SCORECARD LOGÍSTICO:
                          TRAIN        VAL       TEST
----------------------------------------------------
AUC                      0.7479     0.7564     0.7474
KS                       0.3656     0.3857     0.3657
Gini                     0.4958     0.5127     0.4949

DIAGNÓSTICO:
  Brecha train-val AUC: -0.0085 ✅ Sin overfitting


In [5]:
# Verificar NaN en el scorecard
print("NaN en X_train_sc:")
nulos = X_train_sc.isnull().sum()
nulos_existentes = nulos[nulos > 0]
print(f"Variables con NaN: {len(nulos_existentes)}")
print(nulos_existentes.to_string())

NaN en X_train_sc:
Variables con NaN: 18
OWN_CAR_AGE                   142118
EXT_SOURCE_1                  121446
EXT_SOURCE_2                     475
EXT_SOURCE_3                   42755
OBS_30_CNT_SOCIAL_CIRCLE         712
DEF_30_CNT_SOCIAL_CIRCLE         712
OBS_60_CNT_SOCIAL_CIRCLE         712
DEF_60_CNT_SOCIAL_CIRCLE         712
DAYS_LAST_PHONE_CHANGE             1
AMT_REQ_CREDIT_BUREAU_DAY      29071
AMT_REQ_CREDIT_BUREAU_MON      29071
AMT_REQ_CREDIT_BUREAU_QRT      29071
AMT_REQ_CREDIT_BUREAU_YEAR     29071
ANTIGUEDAD_LABORAL_AÑOS        38783
EXT_SOURCE_PROMEDIO              129
EXT_SOURCE_MIN                   129
DTI_X_EXT2                       475
RIESGO_EDAD_SCORE                129
